# Individual Sound Analysis — Crowd Sound Project
Waveform, spectrogramme et paramètres acoustiques pour chaque son.

In [3]:
import subprocess
subprocess.run(["pip", "install", "librosa"], capture_output=True)
!pip install librosa
print("✓ librosa installé")

✓ librosa installé


In [5]:
import os
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, Audio
import librosa
import warnings
warnings.filterwarnings('ignore')

ModuleNotFoundError: No module named 'librosa'

In [ ]:
outputs_folder = "../outputs"
data = []
for sound_folder in os.listdir(outputs_folder):
    json_path = os.path.join(outputs_folder, sound_folder, "results.json")
    if os.path.exists(json_path):
        with open(json_path, "r") as f:
            result = json.load(f)
            result["group"] = sound_folder[0]
            data.append(result)

df = pd.DataFrame(data)
audio_dir = os.path.abspath("../data/sounds")
df["audio_path"] = df["file"].apply(lambda x: os.path.join(audio_dir, x))
print(f"✓ {len(df)} sons chargés")

In [ ]:
def show_sound_details(filename):
    path = df[df["file"] == filename]["audio_path"].values[0]
    y, sr = librosa.load(path, sr=None)
    times = np.linspace(0, len(y)/sr, len(y))
    D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
    freqs = librosa.fft_frequencies(sr=sr)
    frames = librosa.frames_to_time(np.arange(D.shape[1]), sr=sr)

    fig = make_subplots(rows=2, cols=1,
                        subplot_titles=(f"Waveform — {filename}", "Spectrogramme"),
                        vertical_spacing=0.15)
    fig.add_trace(go.Scatter(x=times, y=y, mode="lines",
                             line=dict(color="#1f77b4", width=0.8)), row=1, col=1)
    fig.add_trace(go.Heatmap(z=D, x=frames, y=freqs,
                             colorscale="Viridis", colorbar=dict(title="dB")), row=2, col=1)
    fig.update_layout(title=f"Analyse : {filename}", height=600, showlegend=False)
    fig.update_xaxes(title_text="Temps (s)", row=1, col=1)
    fig.update_xaxes(title_text="Temps (s)", row=2, col=1)
    fig.update_yaxes(title_text="Amplitude", row=1, col=1)
    fig.update_yaxes(title_text="Fréquence (Hz)", row=2, col=1)
    fig.show()

def show_sound_params(filename):
    row = df[df["file"] == filename].iloc[0]
    params = {
        "Paramètre": ["Durée (s)", "F0 moyen (Hz)", "Variabilité F0",
                      "Intensité (RMS)", "Centroïde spectral (Hz)",
                      "Largeur de bande (Hz)", "Rolloff spectral (Hz)"],
        "Valeur": [
            f"{row['duration']:.3f}", f"{row['f0_mean']:.2f}",
            f"{row['f0_variability']:.2f}", f"{row['global_intensity']:.5f}",
            f"{row['spectral_centroid_mean']:.2f}",
            f"{row['spectral_bandwidth_mean']:.2f}",
            f"{row['spectral_rolloff_mean']:.2f}"
        ]
    }
    fig = go.Figure(data=[go.Table(
        header=dict(values=["Paramètre", "Valeur"],
                    fill_color="#1f77b4", font=dict(color="white", size=13), align="left"),
        cells=dict(values=[params["Paramètre"], params["Valeur"]],
                   fill_color="lavender", align="left", font=dict(size=12))
    )])
    fig.update_layout(title=f"Paramètres — {filename} (Groupe {row['group']})", height=300)
    fig.show()

In [ ]:
audio_out = widgets.Output()
file_selector = widgets.Dropdown(options=df["file"].tolist(), description="Son")
button = widgets.Button(description="▶ Jouer le son")

def on_click(b):
    path = df[df["file"] == file_selector.value]["audio_path"].values[0]
    with audio_out:
        audio_out.clear_output()
        display(Audio(path, autoplay=True))

def on_file_change(change):
    if change["name"] == "value":
        show_sound_params(change["new"])
        show_sound_details(change["new"])

button.on_click(on_click)
file_selector.observe(on_file_change)

display(file_selector, button, audio_out)
show_sound_params(file_selector.value)
show_sound_details(file_selector.value)